In [1]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
from collections import deque, defaultdict
import os
import scipy as stats

In [2]:
df1 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump1.parquet')
df2 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump2.parquet')
df3 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump3.parquet')
df4 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump4.parquet')
df5 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump5.parquet')
df6 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump6.parquet')
df7 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump7.parquet')

dump1 = df1.to_pandas()
dump2 = df2.to_pandas()
dump3 = df3.to_pandas()
dump4 = df4.to_pandas()
dump5 = df5.to_pandas()
dump6 = df6.to_pandas()
dump7 = df7.to_pandas()

In [3]:
files_pathname = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\thresholds_test"
"""masq_files = [
    f for f in os.listdir(files_pathname)
    if f.startswith("dump6-masq-") and f.endswith(".parquet")
]

masq_ids = [f.split('-')[-1].removesuffix('.parquet') for f in masq_files]
    #masq_ids = ["112h","113h","200h"]
print(f"Found {len(masq_ids)} masquerade files.")
print("Masquerade IDs in discovery order:", masq_ids)"""

'masq_files = [\n    f for f in os.listdir(files_pathname)\n    if f.startswith("dump6-masq-") and f.endswith(".parquet")\n]\n\nmasq_ids = [f.split(\'-\')[-1].removesuffix(\'.parquet\') for f in masq_files]\n    #masq_ids = ["112h","113h","200h"]\nprint(f"Found {len(masq_ids)} masquerade files.")\nprint("Masquerade IDs in discovery order:", masq_ids)'

In [4]:
def detect_nominal_periods(benign_dumps, candidate_periods=[10, 20, 100, 200, 1000, 2000]):
    """
    Automatically detect the nominal period for each CAN ID from benign data.
    
    Args:
        benign_dumps: List of (name, dataframe) tuples
        candidate_periods: Common nominal periods in ms (from paper)
    
    Returns:
        nominal_period_map: Dict {arb_id: detected_nominal_period}
    """
    
    # Collect all intervals for each ID
    intervals_per_id = defaultdict(list)
    
    for dump_name, dump in benign_dumps:
        print(f"Analyzing {dump_name}...")
        
        d = dump.copy()
        
        # Ensure timestamp is a column
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
            else:
                d = d.reset_index().rename(columns={"index": "timestamp"})
        
        # Sort by timestamp
        d = d.sort_values('timestamp')
        
        # Track previous timestamp per ID
        prev_timestamp = {}
        
        for row in d.itertuples():
            curr_time = row.timestamp
            curr_aid = row.arbitration_id
            
            if curr_aid in prev_timestamp:
                # Calculate interval
                interval = curr_time - prev_timestamp[curr_aid]
                
                # Convert to milliseconds
                if isinstance(interval, pd.Timedelta):
                    interval_ms = interval.total_seconds() * 1000
                elif isinstance(interval, np.timedelta64):
                    interval_ms = float(interval / np.timedelta64(1, 'ms'))
                else:
                    interval_ms = float(interval)
                intervals_per_id[curr_aid].append(interval_ms)
            
            prev_timestamp[curr_aid] = curr_time
    
    # Determine nominal period for each ID
    nominal_period_map = {}
    
    print("\n" + "="*80)
    print("Detected Nominal Periods:")
    print("="*80)
    
    for arb_id, intervals in intervals_per_id.items():
        if len(intervals) < 100:
            print(f"{arb_id}: Insufficient data ({len(intervals)} samples)")
            continue
        
        intervals_array = np.array(intervals)
        
        # Filter out extreme outliers using 5th and 95th percentiles, i tried to do it also for threshold and detection but was not the best idea
        
        """lower_bound = np.percentile(intervals_array, 5)
        upper_bound = np.percentile(intervals_array, 95)
        filtered_intervals = intervals_array[
            (intervals_array >= lower_bound) & 
            (intervals_array <= upper_bound)
        ]"""
        
        # Use median (robust to outliers)
        median_interval = np.median(intervals_array)
        
        """# Method 2: Use mode (most common value)
        # Round to nearest ms to find mode
        intervals_rounded = np.round(intervals_array).astype(int)
        counts = np.bincount(intervals_rounded)
        mode_interval = np.argmax(counts)"""
        
        # Find closest candidate period
        #closest_candidate = min(candidate_periods, 
                               #key=lambda x: abs(x - median_interval))
        
        # Decide which to use:
        # If median is close to a candidate period (within 10%), use candidate
        # Otherwise, use the actual median
        # NEW CODE (always use actual median):
        nominal_period = median_interval  # ← Use actual median, no snapping!
        #method = "median"
        
        nominal_period_map[arb_id] = nominal_period
        
        # Print statistics
        print(f"\n{arb_id}:")
        print(f"  Total samples: {len(intervals)}")
        print(f"  Filtered samples: {len(intervals_array)} (removed {len(intervals) - len(intervals_array)})")
        print(f"  Median (filtered): {median_interval:.2f} ms")
        print(f"  Mean (filtered): {np.mean(intervals_array):.2f} ms")
        print(f"  Std Dev (filtered): {np.std(intervals_array):.2f} ms")
        print(f"  Range (filtered): [{np.min(intervals_array):.2f}, {np.max(intervals_array):.2f}]")
        print(f"  → Selected nominal period: {nominal_period:.2f} ms")
            
    return nominal_period_map

In [5]:
def compute_thresholds(benign_dumps,nominal_period_map, K=5, outlier_percentile=1):
    #threshold_per_id = {}
    residuals_per_id = defaultdict(list)
    
    KNOWN_PE_IDS = {1322, 1314, 1363, 66, 68, 897, 1345, 1356}
    
    for dump_name, dump in benign_dumps:
        print(f"Processing {dump_name}...")
        
        d = dump.copy()
        
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
        
        d = d.sort_values('timestamp')
        
        # Track previous timestamp per ID
        prev_timestamp = {}
        
        for row in d.itertuples():
            curr_time = row.timestamp
            curr_aid = row.arbitration_id
            
            # Get nominal period
            nominal_period = nominal_period_map.get(curr_aid)
            if nominal_period is None:
                continue
                
            if curr_aid in prev_timestamp:
                # Calculate time difference
                time_diff = curr_time - prev_timestamp[curr_aid]
                
                # Convert to milliseconds
                if isinstance(time_diff, pd.Timedelta):
                    time_diff_ms = time_diff.total_seconds() * 1000
                elif isinstance(time_diff, np.timedelta64):
                    time_diff_ms = float(time_diff / np.timedelta64(1, 'ms'))
                else:
                    time_diff_ms = float(time_diff)
                
                is_valid_sample = True
                
                # RULE 1: General "Impossible Speed" Filter (Catches Bursts)
                # If it's faster than 50% of nominal, it's noise/event.
                if time_diff_ms < (nominal_period * 0.5):
                    is_valid_sample = False
                
                # RULE 2: Explicit Strictness for Known PE IDs
                # If this is a known "problem ID", we apply extra strict bounds.
                # We ignore anything that deviates by more than 50% from the mean.
                # This kills the "Long Gaps".
                if curr_aid in KNOWN_PE_IDS:
                    deviation = abs(time_diff_ms - nominal_period)
                    if deviation > (nominal_period * 0.5):
                        is_valid_sample = False
                
                # RULE 3: General "Massive Gap" Filter (Safety Net)
                # If ANY message is > 4x its period, it's likely a dropout/suspension
                # artifact in the training data. Ignore it.
                if time_diff_ms > (nominal_period * 4.0):
                    is_valid_sample = False

                # Update Timestamp & Store (if valid)
                if is_valid_sample:
                    residual = time_diff_ms - nominal_period
                    residuals_per_id[curr_aid].append(residual)
                
            prev_timestamp[curr_aid] = curr_time
            
    # Compute thresholds
    thresholds_per_id = {}
    
    for arb_id, residuals in residuals_per_id.items():
        residuals_array = np.array(residuals)
        
        #lower_bound = np.percentile(residuals_array, outlier_percentile)
        #upper_bound = np.percentile(residuals_array, 100 - outlier_percentile)
        
        # Keep only residuals within [1st, 99th] percentile
        #filtered_residuals = residuals_array[
         #   (residuals_array >= lower_bound) & 
         #   (residuals_array <= upper_bound)
        #]
        
        
        # Calculate statistics
        mu = np.mean(residuals_array)
        sigma = np.std(residuals_array)
        
        # Calculate thresholds: thr = K × σ + μ
        thr_upper = K * sigma + mu
        thr_lower = -K * sigma + mu
        
        thresholds_per_id[arb_id] = {
            'lower': thr_lower,
            'upper': thr_upper,
            'mu': mu,
            'sigma': sigma,
            'K': K,
            'n_samples': len(residuals),
            'n_samples_filtered': len(residuals_array),
            'outliers_removed': len(residuals) - len(residuals_array),
            'nominal_period': nominal_period_map[arb_id]
        }
        
        #Print problematic IDs to verify they are fixed
        if arb_id in KNOWN_PE_IDS:
            print(f"\n[FIXED] ID {arb_id}:")
            print(f"  σ: {sigma:.4f} (Should be small now!)")
            print(f"  Thr: [{thr_lower:.2f}, {thr_upper:.2f}]")
        
        print(f"\n{arb_id}:")
        print(f"  Nominal period: {nominal_period_map[arb_id]:.2f} ms")
        print(f"  Samples: {len(residuals)}")
        print(f"  μ (mean): {mu:.4f} ms")
        print(f"  σ (std dev): {sigma:.4f} ms")
        print(f"  Threshold range: [{thr_lower:.4f}, {thr_upper:.4f}] ms")
    
    return thresholds_per_id

In [6]:
def detect_attacks_first_flag(attack_files, nominal_period_map, thresholds_per_id):
    """
    Detect attacks in test files using trained thresholds.
    
    Returns:
        results: Dict with detection statistics per file
    """
    
    results = {}
    
    for fuzz_intensity in attack_files:
        print(f"\n{'='*80}")
        print(f"Testing: dump6-fuzz-{fuzz_intensity}")
        print(f"{'='*80}")
        
        try:
            # Load attack file
            attack_file = os.path.join(files_pathname, f"dump6-fuzz-{fuzz_intensity}.parquet")
            attack_df = pq.read_table(attack_file).to_pandas()
            
            print(f"Loaded: {len(attack_df):,} messages")
            
            # Ensure timestamp is a column
            if "timestamp" not in attack_df.columns:
                if attack_df.index.name == "timestamp":
                    attack_df = attack_df.reset_index()
            
            attack_df = attack_df.sort_values('timestamp')
            
            # Detection state
            prev_timestamp = {}
            detections = []  # List of (timestamp, arb_id, gradient, detection)
            packets_per_id = defaultdict(int)
            first_detection_per_id = {}  #Track if i already detected attack for this ID
            packet_number = 0  #Track packet index
            
            for row in attack_df.itertuples():
                packet_number += 1
                curr_time = row.timestamp
                curr_aid = row.arbitration_id
                
                # Get nominal period
                nominal_period = nominal_period_map.get(curr_aid)
                if nominal_period is None:
                    continue
                
                # Get thresholds
                thr_info = thresholds_per_id.get(curr_aid)
                if thr_info is None:
                    continue
                
                if curr_aid in prev_timestamp:
                    packets_per_id[curr_aid] += 1
                    # Calculate time difference
                    time_diff = curr_time - prev_timestamp[curr_aid]
                    
                    # Convert to milliseconds
                    if isinstance(time_diff, pd.Timedelta):
                        time_diff_ms = time_diff.total_seconds() * 1000
                    elif isinstance(time_diff, np.timedelta64):
                        time_diff_ms = float(time_diff / np.timedelta64(1, 'ms'))
                    else:
                        time_diff_ms = float(time_diff)
                        
                    #Skip first N packets per ID for warm-up to check if thats the problem
                    if packets_per_id[curr_aid] <= 20:
                        prev_timestamp[curr_aid] = curr_time
                        continue  
                    
                    # ==========================================
                    # INSERT MODIFICATION HERE
                    # ==========================================
                    # Handle Periodic-and-On-Event (PE) messages
                    # If the message arrived WAY too fast (e.g. < 50% of nominal period),
                    # it is likely an event-triggered message, not an attack.
                    if time_diff_ms < (nominal_period * 0.5): 
                        # Update timestamp so the next interval is calculated correctly
                        prev_timestamp[curr_aid] = curr_time
                        continue # Skip detection for this specific packet
                    # ==========================================
                    
                    
                    # Calculate gradient (residual)
                    gradient = time_diff_ms - nominal_period
                    
                    # Check against thresholds
                    thr_lower = thr_info['lower']
                    thr_upper = thr_info['upper']
                    
                    
                    # Detection decision
                    detected = False
                    #attack_type = None
                    if gradient < thr_lower:
                        #detection = "ATTACK_INJECTION"  # Interval too short
                        if curr_aid not in first_detection_per_id:
                            detected = True
                            first_detection_per_id[curr_aid] = packet_number  # Store when first detected
                            print(f"Attack detected ID {curr_aid}")
                            print(f"Packet #{packet_number} at timestamp {curr_time}")
                            print(f"Gradient: {gradient:.2f} ms < Threshold: {thr_lower:.2f} ms")
                    elif gradient > thr_upper:
                        #detection = "ATTACK_SUSPENSION"  # Interval too long
                        if curr_aid not in first_detection_per_id:
                            detected = True
                            first_detection_per_id[curr_aid] = packet_number  # Store when first detected
                            print(f"Attack detected ID {curr_aid}")
                            print(f"Packet #{packet_number} at timestamp {curr_time}")
                            print(f"Gradient: {gradient:.2f} ms > Threshold: {thr_upper:.2f} ms")
                    
                    # Store detection result
                    detections.append({
                        'packet_number': packet_number,
                        'timestamp': curr_time,
                        'arb_id': curr_aid,
                        'gradient': gradient,
                        'thr_lower': thr_lower,
                        'thr_upper': thr_upper,
                        'detected': detected,
                        'label': getattr(row, 'label', None)
                    })
                
                
                #increment packet number to keep track of which is the first attacked packet so i can check manually
                #packet_number += 1

                # Update previous timestamp
                prev_timestamp[curr_aid] = curr_time
            # Calculate statistics
            # Calculate statistics
            # Calculate statistics
            detections_df = pd.DataFrame(detections)
            
            # Get ground truth: which IDs are actually attacked?
            attacked_ids_ground_truth = set(detections_df[detections_df['label'] == 1]['arb_id'].unique())
            
            # Get which IDs you detected
            detected_ids = set(first_detection_per_id.keys())
            
            # CORRECTED: Check timing for True Positive vs False Positive
            true_positive_ids = set()
            false_positive_ids = set()
            
            for aid in detected_ids:
                detected_at = first_detection_per_id[aid]
                
                if aid in attacked_ids_ground_truth:
                    # Check if we detected at or after the attack started
                    gt_packets = detections_df[(detections_df['arb_id'] == aid) & (detections_df['label'] == 1)]
                    gt_first = gt_packets.iloc[0]['packet_number']
                    
                    if detected_at >= gt_first:
                        true_positive_ids.add(aid)  # Detected at/after attack
                    else:
                        false_positive_ids.add(aid)  # Detected BEFORE attack = FP
                else:
                    false_positive_ids.add(aid)  # ID was never attacked = FP
            
            # False Negatives: attacked IDs we never detected (or detected too early)
            false_negative_ids = attacked_ids_ground_truth - true_positive_ids
            
            # Calculate rates
            total_attacked_ids = len(attacked_ids_ground_truth)
            detection_rate = len(true_positive_ids) / total_attacked_ids if total_attacked_ids > 0 else 0
            false_negative_rate = len(false_negative_ids) / total_attacked_ids if total_attacked_ids > 0 else 0
            
            # FPR: use total IDs in dataset as denominator
            total_ids = len(detections_df['arb_id'].unique())
            false_positive_rate = len(false_positive_ids) / total_ids if total_ids > 0 else 0
            
            # Packet-level metrics (simpler)
            #total_attack_packets = (detections_df['label'] == 1).sum()
            #total_benign_packets = (detections_df['label'] == 0).sum()
            
            results[fuzz_intensity] = {
                'total_packets': len(detections),
                'attacked_ids': len(attacked_ids_ground_truth),
                'detected_ids': len(detected_ids),
                'true_positives': len(true_positive_ids),
                'false_positives': len(false_positive_ids),
                'false_negatives': len(false_negative_ids),
                'true_positive_ids': true_positive_ids,     
                'false_positive_ids': false_positive_ids,   
                'false_negative_ids': false_negative_ids,    
                'TPR': detection_rate,
                'FPR': false_positive_rate,
                'FNR': false_negative_rate,
                'first_detections': first_detection_per_id,
            }
        except Exception as e:
            print(f"Error processing file dump6-fuzz-{fuzz_intensity}: {e}")           
    return results

In [7]:
def detect_attacks_hybrid(files_path, nominal_period_map, thresholds_per_id):
    """
    Detect attacks using Hybrid Metrics:
    1. FPR: Packet-based (Total False Alarms / Total Benign Packets) -> Matches Paper
    2. TPR: ID-based (Unique Attack IDs Detected / Total Attack IDs) -> Matches Logical Goal
    """
    attack_files = [f for f in os.listdir(files_path ) 
                    if f.startswith("dump6-") and f.endswith(".parquet")]
    
    results = {}
    
    for file in attack_files:
        print(f"\n{'='*80}")
        
        file_key = file.replace(".parquet", "")
        # Load attack file
        attack_file = os.path.join(files_path, file)
        attack_df = pq.read_table(attack_file).to_pandas()
        
        print(f"Loaded: {len(attack_df):,} messages")
        
        # Ensure timestamp is a column
        if "timestamp" not in attack_df.columns:
            if attack_df.index.name == "timestamp":
                attack_df = attack_df.reset_index()
        
        #attack_df = attack_df.sort_values('timestamp')
        
        # --- STATE VARIABLES ---
        prev_timestamp = {}
        packets_per_id = defaultdict(int)
        
        # --- METRIC COUNTERS ---
        total_benign_packets = 0
        total_false_alarms = 0
        
        total_tp_packets = 0    # TP (Packet level) - NEW
        total_fn_packets = 0    # FN (Packet level) - NEW
        
        # Sets for ID-based TPR
        attacked_ids_ground_truth = set()
        true_positive_ids = set()
        # MODIFICATION: Create a dict to store the specific details of each delay
        detection_details = {}
        
        packet_number = 0 
        
        # --- NEW: LATENCY TRACKING ---
        # Maps ID -> {'pkt': int, 'time': float} (When the attack ACTUALLY started)
        gt_attack_starts = {} 
        # Maps ID -> {'pkt_delay': int, 'time_delay': float} (Recorded on first detection)
        detection_latencies = {}
        
        
        if "susp" in file:
            try:
                # Parse ID from filename (e.g., "dump6-susp-044h.parquet")
                id_part = file.split('-')[-1]
                hex_id = id_part.replace('.parquet', '').replace('h', '')
                target_id = int(hex_id, 16)
                
                # Update Ground Truth Set (Denominator)
                attacked_ids_ground_truth.add(target_id)
                
            except Exception as e:
                print(f"Warning: Could not parse GT ID from {file}: {e}")
        
        
        
        for row in attack_df.itertuples():
            packet_number += 1
            curr_time = row.timestamp
            curr_aid = row.arbitration_id
            
            # Get Ground Truth Label (0 = Benign, 1 = Attack)
            # Ensure your parquet has a 'label' column!
            is_attack_packet = (getattr(row, 'label', 0) == 1)
            
            if is_attack_packet:
                attacked_ids_ground_truth.add(curr_aid)
                if curr_aid not in gt_attack_starts:
                    gt_attack_starts[curr_aid] = {
                        'pkt': packet_number,
                        'time': curr_time
                    }
            
            # Get nominal period & thresholds
            nominal_period = nominal_period_map.get(curr_aid)
            thr_info = thresholds_per_id.get(curr_aid)
            
            if nominal_period is None or thr_info is None:
                continue
            
            if curr_aid in prev_timestamp:
                packets_per_id[curr_aid] += 1
                
                # Calculate time difference
                time_diff = curr_time - prev_timestamp[curr_aid]
                
                # Convert to milliseconds
                if isinstance(time_diff, pd.Timedelta):
                    time_diff_ms = time_diff.total_seconds() * 1000
                elif isinstance(time_diff, np.timedelta64):
                    time_diff_ms = float(time_diff / np.timedelta64(1, 'ms'))
                else:
                    time_diff_ms = float(time_diff)
                    
                """# Warm-up skip
                if packets_per_id[curr_aid] <= 20:
                    prev_timestamp[curr_aid] = curr_time
                    continue"""  
                
                # ==========================================
                # 1. PE MESSAGE FILTER (The Modification)
                # ==========================================
                # If interval is too short (< 50% nominal), it's likely an event-driven 
                # message (legitimate), not an attack because it is too fast, It coule give some problems with the 
                # fuzzing attacks if they are suoper fast so lets try without it= performances go up a lot but also the FPR increases, maybe same problem as for Hamming distance?
                """if time_diff_ms < (nominal_period * 0.5): 
                    prev_timestamp[curr_aid] = curr_time
                    continue """
                # ==========================================
                
                # Calculate gradient
                gradient = time_diff_ms - nominal_period
                is_flagged_lower = (gradient < thr_info['lower'])
                is_flagged_upper = (gradient > thr_info['upper'])
                is_flagged = is_flagged_lower or is_flagged_upper
                
                # ==========================================
                # 2. HYBRID METRIC LOGIC
                # ==========================================
                
                # CASE A: Labeled Attack
                if is_attack_packet:
                    if is_flagged:
                        total_tp_packets += 1
                        true_positive_ids.add(curr_aid)
                        
                        # CALCULATE LATENCY ---
                        # If this is the FIRST time we detect this ID, calculate the delay
                        if curr_aid not in detection_latencies:
                            start_data = gt_attack_starts[curr_aid]
                            
                            # Calculate delays
                            pkt_delay = packet_number - start_data['pkt']
                            
                            detection_details[curr_aid] = {
                                'gt_pkt': start_data['pkt'],      # When attack started
                                'detect_pkt': packet_number,      # When you caught it
                                'delay': pkt_delay                    # The difference
                            }
                            # Time delay requires consistent units (assuming curr_time is float seconds/ms)
                            # If timestamp is datetime, we need total_seconds()
                            try:
                                time_delay = (curr_time - start_data['time'])
                                if hasattr(time_delay, 'total_seconds'):
                                    time_delay = time_delay.total_seconds() * 1000
                            except:
                                time_delay = 0 # Fallback if types mismatch
                                
                            detection_latencies[curr_aid] = {
                                'pkt_delay': pkt_delay,
                                'time_delay': time_delay
                            }
                    else:
                        total_fn_packets += 1
                        
                # CASE B: Labeled Benign (Suspension Heuristic)
                else:
                    if is_flagged:
                        if is_flagged_upper and (time_diff_ms > nominal_period * 50):
                            # Implied Suspension
                            total_tp_packets += 1
                            true_positive_ids.add(curr_aid)
                            
                            # For implied suspension, Latency is harder because we don't have a GT start.
                            # We can skip latency calc or assume latency=0 since it's the first wake-up packet.
                        else:
                            total_false_alarms += 1
                    else:
                        total_benign_packets += 1    
                # ==========================================

            # Update previous timestamp
            prev_timestamp[curr_aid] = curr_time
        
        # --- FINAL CALCULATIONS ---
        
        
        # --- Calculate Metrics ---
        
        # 1. FPR (Packet-Based) - Reliability
        fpr = total_false_alarms / total_benign_packets if total_benign_packets else 0
        
        # 2. TPR & FNR (ID-Based) - Effectiveness
        total_attacked_ids_count = len(attacked_ids_ground_truth)
        tpr = len(true_positive_ids) / total_attacked_ids_count if total_attacked_ids_count else 0
        
        # Identify specifically WHICH IDs were missed
        false_negative_ids = attacked_ids_ground_truth - true_positive_ids
        fnr = len(false_negative_ids) / total_attacked_ids_count if total_attacked_ids_count else 0
        
        # 3. F1 Score (Packet-Based) - Statistical Balance
        precision = total_tp_packets / (total_tp_packets + total_false_alarms) if (total_tp_packets + total_false_alarms) else 0
        recall = total_tp_packets / (total_tp_packets + total_fn_packets) if (total_tp_packets + total_fn_packets) else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0
        
        # 4. Latency
        avg_pkt_delay = np.mean([d['pkt_delay'] for d in detection_latencies.values()]) if detection_latencies else 0
        
        results[file_key] = {
            # Metrics
            'TPR_ID': tpr,
            'FNR_ID': fnr,            # <--- Added back
            'FPR_PKT': fpr,
            'F1_Score': f1,
            
            # Counts
            'TP_Count': len(true_positive_ids),
            'FN_Count': len(false_negative_ids), # <--- Added back
            'FP_Count': total_false_alarms,
            'Total_Attacked_IDs': total_attacked_ids_count, # <--- Added back
            
            # Debugging / Analysis
            'false_negative_ids': false_negative_ids, # <--- Added back
            'Avg_Latency_Pkts': avg_pkt_delay,
            'Latencies': detection_latencies,
            
            # MODIFICATION: Add the details dictionary to results
            'details': detection_details
        }

    return results

In [8]:
from scipy import stats

def find_optimal_K(target_fpr=0.0001):
    """
    Find K that gives the target False Positive Rate using the Empirical Rule.
    
    For normal distribution, FPR = 2 * (1 - CDF(K))
    
    Args:
        target_fpr: Desired false positive rate (e.g., 0.0001 = 0.01%)
    
    Returns:
        K value
    """
    # For two-tailed test, each tail has FPR/2
    tail_prob = target_fpr / 2
    
    # Find K where P(Z > K) = tail_prob
    K = stats.norm.ppf(1 - tail_prob)
    
    return K

In [9]:
def find_pe_candidates(benign_dumps):
    print(f"\n{'='*60}")
    print("PE MESSAGE CANDIDATE SEARCH")
    print(f"{'='*60}")
    print(f"{'ID':<10} {'Median(ms)':<12} {'Min(ms)':<10} {'Ratio':<10} {'Verdict'}")
    print(f"{'-'*60}")
    
    # Aggregate all intervals per ID
    intervals_per_id = defaultdict(list)
    # Ensure timestamp is a column
    
    
    for dump_name, dump in benign_dumps:
        
        d = dump.copy()
        
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
            else:
                d = d.reset_index().rename(columns={"index": "timestamp"})
        
        #d = dump.sort_values('timestamp')
        prev_timestamp = {}
        
        for row in d.itertuples():
            curr_time = row.timestamp
            curr_aid = row.arbitration_id
            
            if curr_aid in prev_timestamp:
                diff = curr_time - prev_timestamp[curr_aid]
                # Convert to ms (assuming float timestamp in seconds)
                # Adjust logic if your timestamp is different
                if hasattr(diff, 'total_seconds'):
                    diff = diff.total_seconds() * 1000
                else:
                    diff = float(diff)
                
                intervals_per_id[curr_aid].append(diff)
            prev_timestamp[curr_aid] = curr_time

    # Analyze
    pe_candidates = []
    for aid, intervals in intervals_per_id.items():
        if len(intervals) < 100: continue
        
        arr = np.array(intervals)
        median = np.median(arr)
        minimum = np.min(arr)
        
        # Logic: If the minimum interval is less than 10% of the median, 
        # it implies a mechanism other than "clock jitter" caused the transmission.
        ratio = minimum / median if median > 0 else 0
        
        if ratio < 0.2: # Threshold: Min is < 20% of Median
            verdict = "🔥 HIGH PROBABILITY PE"
            pe_candidates.append(aid)
        elif ratio < 0.5:
            verdict = "⚠️ Possible PE"
        else:
            verdict = "Periodic"
            
        if ratio < 0.5: # Only print interesting ones
            print(f"{aid:<10} {median:<12.2f} {minimum:<10.2f} {ratio:<10.2f} {verdict}")
            
    return pe_candidates


benign_dumps = [
        ("dump1", dump1),
        ("dump2", dump2),
        ("dump3", dump3),
        ("dump4", dump4),
        ("dump5", dump5),
        ("dump6", dump6),
        #("dump7", dump7)
    ]
# Run this in your main block
pe_ids = find_pe_candidates(benign_dumps)
print(f"Recommended Filter List: {pe_ids}")



PE MESSAGE CANDIDATE SEARCH
ID         Median(ms)   Min(ms)    Ratio      Verdict
------------------------------------------------------------
897        20.00        6.48       0.32       ⚠️ Possible PE
1345       99.45        23.76      0.24       ⚠️ Possible PE
1322       42.51        5.53       0.13       🔥 HIGH PROBABILITY PE
1356       200.00       90.34      0.45       ⚠️ Possible PE
1314       199.92       29.79      0.15       🔥 HIGH PROBABILITY PE
1363       199.92       25.24      0.13       🔥 HIGH PROBABILITY PE
66         997.59       10.19      0.01       🔥 HIGH PROBABILITY PE
68         997.58       10.23      0.01       🔥 HIGH PROBABILITY PE
Recommended Filter List: [1322, 1314, 1363, 66, 68]


In [10]:
#LEts do the training part for now, i have to set the thresholds
if __name__ == "__main__":
    
    #they say to compute onlty from idling data but dump6 which is the one with the attacks is in motion, i suppose i will use everything in motion then
    benign_dumps = [
        ("dump1", dump1),
        ("dump2", dump2),
        ("dump3", dump3),
        ("dump4", dump4),
        ("dump5", dump5),
        ("dump6", dump6),
        #("dump7", dump7)
    ]
    
    
    nominal_period_map = detect_nominal_periods(benign_dumps)
    #frames = [dump1, dump2, dump3, dump4, dump5, dump6, dump7]
    
    cumulative_residuals_per_id = defaultdict(list)
    #nominal_periods = [10, 20 ,100, 200, 1000] # nominal periods in ms
    fuzz_intensity = ["10", "20", "30", "40", "50", "60", "70", "80", "90", "100", "200", "300", "400", "500", "1000", "1500", "2000"]
    """for dump_name, dump in benign_dumps:
        
        d= dump.copy()
        #ensure timestamp is a column
        if "timestamp" not in d.columns:
                if d.index.name == "timestamp":
                    d = d.reset_index()
                else:
                    d = d.reset_index().rename(columns={"index": "timestamp"})
        d = d.sort_values("timestamp")
        
        #previous timestamp and cumulative residuals
        prev_timestamp = {}
        cumulative_residuals_current_dump = {}
        
        #R_cum_k = 0            
        #now for each row calculate the residuals
        for row in  d.itertuples():
            curr_time = row.timestamp
            curr_aid = row.arbitration_id
            
            # get current nominal period
            nominal_period = nominal_period_map.get(curr_aid)
            if nominal_period is None:
                continue 
            
            # Initialize cumulative residual for this ID in this dump
            if curr_aid not in cumulative_residuals_current_dump:
                cumulative_residuals_current_dump[curr_aid] = 0
            
            if curr_aid in prev_timestamp:
                time_diff = curr_time - prev_timestamp[curr_aid]
                
                #update with current timestamp
                prev_timestamp[curr_aid] = curr_time
                
                #now calculate the ith residual for a nominal period, TO DO i have to convert the time_diff into a float or python format because rn it gives an error
                if isinstance(time_diff, pd.Timedelta):
                    time_diff_ms = time_diff.total_seconds() * 1000
                elif isinstance(time_diff, np.timedelta64):
                    time_diff_ms = float(time_diff / np.timedelta64(1, 'ms'))
                else:
                    time_diff_ms = float(time_diff)
                r_i = time_diff_ms - nominal_period
                
                #now the cumulative residual
                cumulative_residuals_current_dump[curr_aid] += r_i
                
                #Store this cumulative value
                cumulative_residuals_per_id[curr_aid].append(
                    cumulative_residuals_current_dump[curr_aid]
                )
            #update with current timestamp
            prev_timestamp[curr_aid] = curr_time"""
            
    """target_fpr = 0.0001  # 0.01% false positive rate
    K = find_optimal_K(target_fpr)
    print(f"Using K = {K} for target FPR = {target_fpr*100:.4f}%")"""

    print(f"\n{'='*80}")
    print("COMPUTING THRESHOLDS")   
    #TO-DO for each cumulative residual i have to calculate mean and std deviation to set thresholds, have to understand why better        
    thresholds_per_id = compute_thresholds(benign_dumps, nominal_period_map, K=5, outlier_percentile=5)    
    
    thresholds_df = pd.DataFrame.from_dict(thresholds_per_id, orient='index')
    thresholds_df.to_csv('ErrIDS_artifacts/thresholds.csv')
    
    # Check results
    for arb_id, residuals in cumulative_residuals_per_id.items():
        print(f"\nID {arb_id}:")
        print(f"  Number of samples: {len(residuals)}")
        print(f"  Range: [{min(residuals):.2f}, {max(residuals):.2f}]")
        print(f"  First 5 values: {residuals[:5]}")
        
    
    # Now thresholds_per_id contains the thresholds for each CAN ID so i can try and do the detection
    attack_results = detect_attacks_hybrid(files_pathname, nominal_period_map, thresholds_per_id)
    
    """print(f"\n{'='*80}")
    print("ATTACK DETECTION SUMMARY")
    print(f"{'='*80}")
    print(f"{'File':<20} {'Unique IDs':<12} {'First Detected At':<20}")
    print(f"{'-'*80}")

    for fuzz_intensity, result in attack_results.items():
        print(f"{fuzz_intensity:<20} {result['detection_rate_per_id']*100:>13.2f}% {result['FPR_per_id']*100:>10.2f}% {len(result['detected_ids']):<12}")

    print(f"\n{'='*80}")
    print("FIRST DETECTION PER ID")
    print(f"{'='*80}")
    print(f"{'File':<20} {'ID':<12} {'First Detected At':<20} {'Ground Truth':<20}")
    print(f"{'-'*80}")

    for fuzz_intensity, result in attack_results.items():
        for aid, pkt_num in result['first_detections'].items():
            print(f"{fuzz_intensity:<20} {aid:<12} Packet #{pkt_num}")"""
            
            
            
    print(f"\n{'='*80}")
    print("ATTACK DETECTION SUMMARY (UNIFIED + LATENCY)")
    print(f"{'='*80}")
    # Added FNR and Total Attacked columns
    print(f"{'File':<10} {'TPR':<8} {'FNR':<8} {'FPR':<9} {'F1':<8} {'TP':<4} {'FN':<4} {'Tot':<4} {'Avg Delay':<12}")
    print(f"{'-'*80}")

    for filename, result in attack_results.items():
        print(f"{filename:<10} "
              f"{result['TPR_ID']*100:>6.2f}% "
              f"{result['FNR_ID']*100:>6.2f}% "
              f"{result['FPR_PKT']*100:>7.4f}% "
              f"{result['F1_Score']*100:>6.2f}% "
              f"{result['TP_Count']:<4} "
              f"{result['FN_Count']:<4} "
              f"{result['Total_Attacked_IDs']:<4} "
              f"{result['Avg_Latency_Pkts']:>10.2f}")

    print(f"\n{'='*80}")
    print("MISSED ATTACKS (DEBUGGING)")
    print(f"{'='*80}")
    for filename, result in attack_results.items():
        if result['false_negative_ids']:
            print(f"{filename:<10} Missed IDs: {result['false_negative_ids']}")
            
    print(f"\n{'='*80}")
    print("DETECTION DELAY DETAILS (Per Attack ID)")
    print(f"{'='*80}")
    print(f"{'File':<10} {'ID':<8} {'GT Start':<10} {'Detected':<10} {'Delay (Pkts)':<12}")
    print(f"{'-'*80}")

    for file, res in attack_results.items():
        # Check if we have details (might be empty if no TPs found)
        details = res.get('details', {})
        
        if not details:
            continue
            
        for aid, info in details.items():
            print(f"{file:<10} {aid:<8} {info['gt_pkt']:<10} {info['detect_pkt']:<10} {info['delay']:<12}")

Analyzing dump1...
Analyzing dump2...
Analyzing dump3...
Analyzing dump4...
Analyzing dump5...
Analyzing dump6...

Detected Nominal Periods:

356:
  Total samples: 1028843
  Filtered samples: 1028843 (removed 0)
  Median (filtered): 10.00 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.20 ms
  Range (filtered): [7.86, 79.97]
  → Selected nominal period: 10.00 ms

688:
  Total samples: 1028706
  Filtered samples: 1028706 (removed 0)
  Median (filtered): 10.00 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.63 ms
  Range (filtered): [6.22, 21.27]
  → Selected nominal period: 10.00 ms

593:
  Total samples: 1028705
  Filtered samples: 1028705 (removed 0)
  Median (filtered): 10.00 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.30 ms
  Range (filtered): [6.87, 20.38]
  → Selected nominal period: 10.00 ms

790:
  Total samples: 1028863
  Filtered samples: 1028863 (removed 0)
  Median (filtered): 9.99 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.85 ms
  Range 

In [11]:
#attack_df = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\thresholds_test\dump6-fuzz-10.parquet').to_pandas()
#attack_df['label'] = attack_df['label'].map({"False":0,"True": 1})
#attack_df.to_csv('reading_fuzz_TF_10.csv', index=False)